# 08_sarima_forecasting.ipynb (Final)

**Objective:** Forecast Bitcoin prices using ARIMA and SARIMA, then save prediction comparisons for final reporting.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load time series data
df = pd.read_csv('../data/bitcoin_timeseries.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').set_index('timestamp')

# Resample hourly
df_hourly = df.resample('1h').mean().dropna()

# Train/test split
train = df_hourly[:-24]
test = df_hourly[-24:]

## 🔹 ARIMA Forecast

In [ ]:
model_arima = auto_arima(train, seasonal=False, trace=True, error_action='ignore', suppress_warnings=True)
forecast_arima = model_arima.predict(n_periods=len(test))

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=1509047.761, Time=20.81 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=1516601.855, Time=1.32 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=1509524.961, Time=1.56 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=1509067.235, Time=3.45 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=1516603.620, Time=0.73 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=1509067.436, Time=16.06 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=1509055.330, Time=4.09 sec
 ARIMA(3,1,2)(0,0,0)[0] intercept   : AIC=1509047.863, Time=30.83 sec
 ARIMA(2,1,3)(0,0,0)[0] intercept   : AIC=1509054.142, Time=8.93 sec


## 🔹 SARIMA Forecast

In [ ]:
model_sarima = auto_arima(train, seasonal=True, m=24, trace=True, error_action='ignore', suppress_warnings=True)
forecast_sarima = model_sarima.predict(n_periods=len(test))

## 📊 Forecast Comparison + Export

In [ ]:
# Generate forecast timestamps
last_timestamp = train.index[-1]
forecast_horizon = min(len(forecast_arima), len(forecast_sarima))
forecast_index = [last_timestamp + timedelta(hours=i) for i in range(1, forecast_horizon + 1)]

# Save DataFrame for 05_reporting
df_comparison = pd.DataFrame({
    'timestamp': forecast_index,
    'arima_pred': list(forecast_arima)[:forecast_horizon],
    'sarima_pred': list(forecast_sarima)[:forecast_horizon]
})
df_comparison.to_csv('../reports/arima_vs_sarima_forecast.csv', index=False)
print('✅ Saved: arima_vs_sarima_forecast.csv')

In [ ]:
# Plot
plt.figure(figsize=(12, 5))
plt.plot(forecast_index, forecast_arima[:forecast_horizon], label='ARIMA', marker='o')
plt.plot(forecast_index, forecast_sarima[:forecast_horizon], label='SARIMA', marker='x')
plt.title('ARIMA vs SARIMA Forecast Comparison')
plt.xlabel('Timestamp')
plt.ylabel('BTC Price (USD)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('../reports/arima_sarima_comparison_plot.png')
plt.show()

In [ ]:
# Save metrics to file
mae_arima = mean_absolute_error(test, forecast_arima)
rmse_arima = mean_squared_error(test, forecast_arima, squared=False)
mae_sarima = mean_absolute_error(test, forecast_sarima)
rmse_sarima = mean_squared_error(test, forecast_sarima, squared=False)

with open('../reports/arima_sarima_metrics.txt', 'w') as f:
    f.write(f'ARIMA MAE: {mae_arima:.2f} USD\n')
    f.write(f'ARIMA RMSE: {rmse_arima:.2f} USD\n')
    f.write(f'SARIMA MAE: {mae_sarima:.2f} USD\n')
    f.write(f'SARIMA RMSE: {rmse_sarima:.2f} USD\n')

print('✅ Metrics saved to arima_sarima_metrics.txt')